<a href="https://colab.research.google.com/github/sarangis/python_learning/blob/main/MP1_Part_B_NB_LLM_Based_Spam_Classification_Gradio_Interface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Certification Programme in Agentic and Generative AI
## A Programme by IISc and TalentSprint
### Mini-Project 1 Part-B: LLM Based Spam Classification and Gradio Interface

## Learning Objectives

At the end of the mini-project, you will be able to :

* Load the trained NN model from Part-A
* Initialize models like `gpt-oss-20b`, `llama-3.1-8b-instant` to use via Groq API key (free-tier)
* Run Ollama server and initialize models like `gemma3:4b`, `llama3.1:8b`
* Experiment with different prompting techniques for spam classification
* Create a Gradio interface to enable users to choose the model to use for response generation


## Problem Statement

Spam messages continue to pose a significant challenge across email and messaging platforms, often leading to security risks, phishing attempts, and unwanted content. Detecting and filtering spam effectively requires robust classification techniques that can adapt to evolving message patterns.

Traditional machine learning models can perform spam detection based on learned patterns in historical data, while modern LLMs offer alternative approaches through prompt-based reasoning and text understanding.

In this project, the objective is to build **a spam classification system** that uses both **a trained neural network model and multiple Large Language Models (LLMs)**.

The project begins by loading the neural network model developed in Part-A as one of the spam classifier. In addition, learners have to configure and use LLMs accessed through the **Groq API** as well as locally hosted models running through **Ollama**, enabling experimentation with different inference environments.

You will also explore **prompt engineering techniques**, including zero-shot and few-shot prompting, to perform spam classification using LLMs and compare their performance with the neural network baseline.

To make the system interactive and user-friendly, a **Gradio-based interface** will be developed that allows users to input a message and select which model (neural network, Groq-hosted LLM, or locally hosted Ollama model) should be used for classification.

The final system will demonstrate how different AI approaches—traditional machine learning and modern LLM-based inference—can be used and evaluated within a single application for solving a real-world **spam detection problem**.

## Dataset

(Used for training the Neural Network model in Part-A)

The [SMS Spam Collection dataset](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset) is a set of SMS tagged messages that have been collected for SMS Spam research. It contains one set of SMS messages in English of 5,574 messages, tagged acording being ham (legitimate) or spam.


## Grading = 6 Points

In [2]:
# prompt: Create a hidden code cell with @#title Download the Dataset. Data should be downloaded from the following link: https://cdn.exec.talentsprint.com/static/aimlops/c3/spam.csv

#@title Download the Dataset
!wget https://cdn.exec.talentsprint.com/static/aimlops/c3/spam.csv


--2026-03-21 13:57:41--  https://cdn.exec.talentsprint.com/static/aimlops/c3/spam.csv
Resolving cdn.exec.talentsprint.com (cdn.exec.talentsprint.com)... 172.105.52.210
Connecting to cdn.exec.talentsprint.com (cdn.exec.talentsprint.com)|172.105.52.210|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 503663 (492K) [application/octet-stream]
Saving to: ‘spam.csv.1’

spam.csv.1          100%[===================>] 491.86K   411KB/s    in 1.2s    

2026-03-21 13:57:43 (411 KB/s) - ‘spam.csv.1’ saved [503663/503663]



### Import Neccesary Packages

In [3]:
# Please feel free to add/remove installations here

# Initial Packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### Load the Trained Neural Network Model from Part-A

Load the neural network model that was trained and saved in Part-A of the project.

This model will be used as one of the spam classifiers to generate predictions and compare its performance with LLM-based approaches.

In [4]:
from tensorflow import keras

# Load the trained Keras model
spam_nn_model = keras.models.load_model('/content/spam_classifier_model.keras')

print("Neural Network model loaded successfully!")
spam_nn_model.summary()

Neural Network model loaded successfully!


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_embedding (Embedding)      │ (None, 250, 128)       │     1,280,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_4      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,939,845 (15.03 MB)

 Trainable params: 1,313,281 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,626,564 (10.02 MB)

### Initialize Models such as `gpt-oss-20b` and `llama-3.1-8b-instant` using the **Groq** API Key (Free Tier) [1 Point]

Configure access to the Groq API using the your Groq API key and initialize the mentioned large language models.

These models will be used to perform spam classification by sending prompts through the Groq inference API.

**Hint:** Refer to the *Supplementary Notebook - Getting Started with Groq API and Ollama Server.ipynb*, released along with this notebook.

In [5]:
# Install the Groq library
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 4.1 MB/s eta 0:00:00


You'll need a Groq API key. If you don't have one, you can obtain it from the [Groq website](https://console.groq.com/keys).

**To securely store your API key in Google Colab:**
1. Click on the '🔑 Secrets' icon on the left sidebar.
2. Click 'Enable secrets' if prompted.
3. Add a new secret named `GROQ_API_KEY` and paste your Groq API key as its value.
4. Make sure 'Notebook access' is toggled on for this secret.

In [6]:
# Import necessary packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Securely load the API key from Colab secrets
from google.colab import userdata
import os

try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    os.environ['GROQ_API_KEY'] = GROQ_API_KEY
    print("Groq API Key loaded successfully.")
except userdata.SecretValueError:
    print("Groq API Key not found in Colab secrets. Please add it as 'GROQ_API_KEY'.")
    GROQ_API_KEY = None

Groq API Key loaded successfully.


In [7]:
from groq import Groq

if GROQ_API_KEY:
    client = Groq(
        api_key=GROQ_API_KEY,
    )

    # Initialize Groq LLMs
    groq_llms = {
        "gpt-oss-20b": client.chat.completions.create,
        "llama-3.1-8b-instant": client.chat.completions.create,
    }
    print("Groq client initialized and LLMs configured.")
else:
    print("Groq API key is not set. Groq LLMs cannot be initialized.")
    groq_llms = {}


Groq client initialized and LLMs configured.


In [ ]:
## Add your prompt and code here

### Run the **Ollama Server** and Initialize Models such as `gemma3:4b` and `llama3.1:8b` [2 Points]

Start the Ollama server locally in Colab and load the required open-source models.

These models will allow local inference for spam classification, enabling experimentation without relying on external APIs.

**Hint:** Refer to the *Supplementary Notebook - Getting Started with Groq API and Ollama Server.ipynb*, released along with this notebook.

*[If you face any compute/space related issues in Google Colab, then you may proceed only with one model - `gemma3:4b`]*

In [10]:
# Install zstd
!sudo apt-get update && sudo apt-get install -y zstd

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in the background
import subprocess
import time

print("Starting Ollama server...")
# Use a different port if 11434 is already in use by a previous attempt
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(10) # Give the server some time to start
print("Ollama server started.")

# Pull models
# If you face any compute/space related issues, you can comment out one of the models
print("Pulling gemma3:4b model...")
!ollama pull gemma3:4b
print("gemma3:4b model pulled.")

print("Pulling llama3.1:8b model...")
!ollama pull llama3.1:8b
print("llama3.1:8b model pulled.")

# Install ollama Python package
!pip install ollama

# Initialize Ollama client
import ollama

ollama_llms = {
    "gemma3:4b": ollama.chat, # ollama.chat can be used for chat completions
    "llama3.1:8b": ollama.chat
}
print("Ollama client initialized and LLMs configured.")

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,939 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,896 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,794 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,303 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,836 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:12 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:14 

ModuleNotFoundError: No module named 'ollama'

### Experiment with Different Prompting Techniques for Spam Classification [1 Point]

Design and test various prompting strategies such as ***zero-shot prompting***, ***few-shot prompting***, and ***instruction-based prompts***.

Evaluate how different prompt structures influence the accuracy and reliability of spam classification using LLMs.

In [ ]:
## Add your prompt and code here

### Create a `Gradio Interface` to Allow Users to Select the Model for Response Generation [2 Points]

Develop an interactive **Gradio interface** (as shown below) where users can input a message and select which model to use for classification.

The interface should support switching between models (e.g., Groq-hosted models, Ollama models, or the trained neural network) and display the predicted result to the user along with the time it took to get the response.

<img src='https://drive.google.com/uc?id=1qgT3Zp08gMwY9a_UT8_kiOG2BdT1Qmdm' width=800px>

In [1]:
import gradio as gr
import time

# Placeholder functions for classification
def classify_with_nn(message):
    """Placeholder for Neural Network classification logic."""
    # Implement your NN classification here
    time.sleep(1) # Simulate processing time
    return f"NN Classification: This is a {'SPAM' if 'urgent' in message.lower() else 'HAM'} message."

def classify_with_groq_llm(message, llm_model):
    """Placeholder for Groq LLM classification logic."""
    # Implement your Groq LLM classification here
    time.sleep(2) # Simulate processing time
    return f"Groq LLM ({llm_model}) Classification: This is a {'SPAM' if 'free money' in message.lower() else 'HAM'} message."

def classify_with_ollama_llm(message, llm_model):
    """Placeholder for Ollama LLM classification logic."""
    # Implement your Ollama LLM classification here
    time.sleep(3) # Simulate processing time
    return f"Ollama LLM ({llm_model}) Classification: This is a {'SPAM' if 'winner' in message.lower() else 'HAM'} message."

def classify_message(message, model_type, groq_model=None, ollama_model=None):
    start_time = time.time()
    result = ""

    if model_type == "Neural Network":
        result = classify_with_nn(message)
    elif model_type == "Groq LLM":
        if groq_model:
            result = classify_with_groq_llm(message, groq_model)
        else:
            result = "Please select a Groq LLM."
    elif model_type == "Ollama LLM":
        if ollama_model:
            result = classify_with_ollama_llm(message, ollama_model)
        else:
            result = "Please select an Ollama LLM."
    else:
        result = "Please select a valid model type."

    end_time = time.time()
    time_taken = round(end_time - start_time, 2)

    return f"{result}\nTime taken: {time_taken} seconds"


# Gradio Interface
with gr.Blocks() as demo:
    gr.Markdown("# LLM Based Spam Classification")
    with gr.Row():
        message_input = gr.Textbox(label="Enter your message here:", placeholder="Type a message to classify...")
    with gr.Row():
        model_type_radio = gr.Radio(
            ["Neural Network", "Groq LLM", "Ollama LLM"],
            label="Select Classification Model",
            value="Neural Network"
        )
    with gr.Row():
        groq_model_dropdown = gr.Dropdown(
            ["gpt-oss-20b", "llama-3.1-8b-instant"],
            label="Select Groq LLM (if 'Groq LLM' selected above)",
            value="gpt-oss-20b",
            interactive=True
        )
        ollama_model_dropdown = gr.Dropdown(
            ["gemma3:4b", "llama3.1:8b"],
            label="Select Ollama LLM (if 'Ollama LLM' selected above)",
            value="gemma3:4b",
            interactive=True
        )
    with gr.Row():
        classify_button = gr.Button("Classify Message")
    with gr.Row():
        output_text = gr.Textbox(label="Classification Result", interactive=False)

    classify_button.click(
        fn=classify_message,
        inputs=[message_input, model_type_radio, groq_model_dropdown, ollama_model_dropdown],
        outputs=output_text
    )

    # Update dropdown visibility based on model type selection
    model_type_radio.change(
        lambda x: [gr.Dropdown(visible=x=='Groq LLM'), gr.Dropdown(visible=x=='Ollama LLM')],
        inputs=model_type_radio,
        outputs=[groq_model_dropdown, ollama_model_dropdown]
    )

# To launch the interface, you can uncomment the line below after defining all models
demo.launch(debug=True, share=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://cb9402259d5e86f376.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://cb9402259d5e86f376.gradio.live


---

<center>
$END$
</center>

---